<a href="https://colab.research.google.com/github/DhimanTarafdar/simple-gk-answring-system-using-rnn/blob/main/simple_gk_answring_system_code_explain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## ১. Dataset লোড করা

```python
!git clone https://github.com/tajuar-akash-hub/Datasets
```

```python
import pandas as pd

df = pd.read_csv("/content/Datasets/gk_qna_dataset.csv")
df
```

```python
df['question']       # শুধু question column দেখা
df['question'][0]    # প্রথম প্রশ্নটি দেখা
```

### 🔍 ব্যাখ্যা

| লাইন | কাজ |
|------|-----|
| `!git clone ...` | GitHub থেকে dataset সহ repository clone করে Google Colab-এ নিয়ে আসে |
| `import pandas as pd` | Data manipulation-এর জন্য pandas library import করা হয় |
| `pd.read_csv(...)` | CSV ফাইলটি পড়ে একটি DataFrame (টেবিল) তৈরি করে `df` ভেরিয়েবলে রাখা হয় |
| `df['question']` | DataFrame-এর শুধু "question" column-টি বের করে দেখা হয় |
| `df['question'][0]` | ০ নম্বর index-এর (প্রথম) প্রশ্নটি দেখা হয় |

**Dataset Structure:** CSV-তে দুটি column আছে — `question` এবং `answer`। প্রতিটি row একটি GK প্রশ্ন এবং তার উত্তর।

---

## ২. Tokenization (টোকেনাইজেশন)

```python
import re

def tokenize(text):

    # ধাপ ১: সব অক্ষর lowercase করা
    text = text.lower()

    # ধাপ ২: কোটেশন চিহ্ন সরানো (" এবং ')
    text = re.sub(r"[\"']", "", text)

    # ধাপ ৩: শুধু অক্ষর (a-z) ও সংখ্যা (0-9) ছাড়া বাকি সব চিহ্ন সরানো
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # ধাপ ৪: একাধিক space কে একটি space বানানো
    text = re.sub(r"\s+", " ", text).strip()

    # ধাপ ৫: শব্দে ভাগ করা (split)
    tokens = text.split()

    return tokens
```

```python
print(tokenize(df['question'][0]))
# উদাহরণ আউটপুট: ['what', 'is', 'the', 'capital', 'of', 'france']
```

### 🔍 ব্যাখ্যা

Tokenization মানে হলো একটি বড় text-কে ছোট ছোট শব্দে (token) ভেঙে দেওয়া। Machine Learning মডেল সরাসরি text বোঝে না, তাই আগে text পরিষ্কার করে শব্দে ভাগ করতে হয়।

| ধাপ | Code | উদ্দেশ্য |
|-----|------|----------|
| ১ | `text.lower()` | "What" এবং "what" কে একই শব্দ হিসেবে গণ্য করতে সব lowercase করা হয় |
| ২ | `re.sub(r"[\"']", ...)` | `"Paris"` বা `'Paris'` থেকে কোটেশন সরানো |
| ৩ | `re.sub(r"[^a-z0-9\s]", ...)` | `?`, `.`, `!` এই ধরনের punctuation সরানো, কারণ এগুলো অর্থ বহন করে না |
| ৪ | `re.sub(r"\s+", ...)` | "hello&nbsp;&nbsp;&nbsp;world" → "hello world" — অতিরিক্ত space পরিষ্কার |
| ৫ | `text.split()` | "what is france" → `['what', 'is', 'france']` — list of words |

**`re` module কী?** Python-এর Regular Expression (regex) library, যা text pattern matching ও replacement করতে ব্যবহার হয়।

---

## ৩. Vocabulary (শব্দভাণ্ডার) তৈরি করা

```python
# UNK মানে Unknown — যে শব্দ vocab-এ নেই তাকে 0 দিয়ে represent করা হবে
vocab = {'<UNK>': 0}
```

```python
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer   = tokenize(row['answer'])

    # question ও answer-এর সব token একসাথে মেলানো
    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)   # নতুন শব্দকে পরবর্তী index দাও
```

```python
df.apply(build_vocab, axis=1)   # প্রতিটি row-এর উপর build_vocab চালানো

print(vocab)          # পুরো vocab dictionary দেখা
print(len(vocab))     # মোট কতটি unique শব্দ আছে
print(vocab['what'])  # 'what' শব্দের index কত
```

### 🔍 ব্যাখ্যা

**Vocabulary কেন দরকার?** Neural Network সংখ্যা ছাড়া কাজ করতে পারে না। তাই প্রতিটি অনন্য শব্দকে একটি unique number (index) দিতে হয়।

| Code | কাজ |
|------|-----|
| `vocab = {'<UNK>': 0}` | Dictionary শুরু করা হয় — `<UNK>` মানে অজানা শব্দ, index = 0 |
| `tokenize(row['question'])` | প্রতিটি প্রশ্নকে token করা |
| `merged_tokens = q_tokens + a_tokens` | প্রশ্ন ও উত্তর দুটোর শব্দ একসাথে ধরা হয় |
| `if token not in vocab` | যদি শব্দটি আগে না থাকে, তাহলে নতুন index দাও |
| `vocab[token] = len(vocab)` | সেই মুহূর্তে vocab-এ যতটি শব্দ আছে, সেটাই নতুন শব্দের index হয় |
| `df.apply(build_vocab, axis=1)` | DataFrame-এর প্রতিটি row-এর উপর function চালানো হয় |

**উদাহরণ:** "What is France?" → `{'<UNK>':0, 'what':1, 'is':2, 'france':3, ...}`

---

## ৪. Text থেকে Index Conversion

```python
def text_to_indices(text, vocab):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])    # vocab-এ থাকলে তার index যোগ করো
        else:
            indexed_text.append(vocab['<UNK>'])  # না থাকলে 0 (UNK) যোগ করো

    return indexed_text
```

```python
text_to_indices("the is Phitron", vocab)
# আউটপুট: [5, 2, 0]  (Phitron vocab-এ নেই তাই 0)
```

### 🔍 ব্যাখ্যা

| Code | কাজ |
|------|-----|
| `for token in tokenize(text)` | text-কে আগে tokenize করো, তারপর প্রতিটি শব্দের জন্য কাজ করো |
| `vocab[token]` | vocab dictionary থেকে ওই শব্দের numeric index বের করা |
| `vocab['<UNK>']` → 0 | অজানা শব্দের জায়গায় 0 বসানো |

**কেন দরকার?** এটি মূলত text → numbers রূপান্তর। RNN model শুধু সংখ্যা নিতে পারে, তাই "What is France?" → `[1, 2, 3]` এই রূপে দিতে হয়।

---

## ৫. Dataset এবং DataLoader তৈরি করা

```python
import torch
from torch.utils.data import Dataset, DataLoader
```

```python
class QADataset(Dataset):

    def __init__(self, df, vocab):
        self.df    = df
        self.vocab = vocab

    def __len__(self):
        # Dataset-এ মোট কতটি sample আছে
        return self.df.shape[0]

    def __getitem__(self, index):
        # নির্দিষ্ট index-এর question ও answer numeric tensor-এ রূপান্তর করে দাও
        numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numerical_answer   = text_to_indices(self.df.iloc[index]['answer'],   self.vocab)

        return torch.tensor(numerical_question), torch.tensor(numerical_answer)
```

```python
dataset    = QADataset(df, vocab)
print(dataset[1])   # index 1-এর (question_tensor, answer_tensor) দেখা

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for question, answer in dataloader:
    print(question, answer[0])
```

### 🔍 ব্যাখ্যা

PyTorch-এ নিজের dataset বানাতে হলে `Dataset` class-টি inherit করতে হয় এবং ৩টি method লিখতে হয়:

| Method | কাজ |
|--------|-----|
| `__init__` | DataFrame ও vocab সংরক্ষণ করে |
| `__len__` | `len(dataset)` করলে মোট sample সংখ্যা দেয় |
| `__getitem__(index)` | নির্দিষ্ট index-এর data ফেরত দেয় (tensor হিসেবে) |

**DataLoader কেন?**
- `batch_size=1` → একটি করে sample মডেলে দেওয়া হবে
- `shuffle=True` → প্রতিটি epoch-এ data এলোমেলো ক্রমে দেওয়া হবে, যাতে মডেল কোনো pattern মুখস্থ না করে
**`torch.tensor(...)`** — Python list-কে PyTorch tensor-এ রূপান্তর করে, যা GPU-তে চলতে পারে।

---

## ৬. Squeeze ও Unsqueeze — Tensor Shape বোঝা

```python
# Squeeze: অতিরিক্ত dimension সরানো
import torch

x = torch.tensor([[[1, 2, 3]]])   # shape: (1, 1, 3)
y = x.squeeze(0)                  # shape: (1, 3) — প্রথম dimension সরানো
print(y.shape)   # torch.Size([1, 3])
```

```python
# Unsqueeze: নতুন dimension যোগ করা
x = torch.tensor([1, 2, 3])   # shape: (3,)
y = x.unsqueeze(0)            # shape: (1, 3) — সামনে একটি batch dimension যোগ
print(y.shape)   # torch.Size([1, 3])
```

### 🔍 ব্যাখ্যা

Neural Network-এ data সবসময় নির্দিষ্ট shape-এ দিতে হয়। Shape ঠিক করতে এই দুটি operation ব্যবহার হয়:

| Operation | কাজ | উদাহরণ |
|-----------|-----|---------|
| `squeeze(dim)` | নির্দিষ্ট dimension-টি সরিয়ে দেয় (যদি সেটি 1 হয়) | `(1,1,3)` → `(1,3)` |
| `unsqueeze(dim)` | নির্দিষ্ট স্থানে নতুন dimension (1) যোগ করে | `(3,)` → `(1,3)` |

**কেন দরকার?** RNN-এর `forward` method-এ `final` tensor-এর shape হয় `(1, 1, hidden_size)`, কিন্তু Linear layer চায় `(1, hidden_size)` — তাই `squeeze(0)` করতে হয়।

---

## ৭. RNN Model Architecture তৈরি করা

```python
import torch.nn as nn

class simpleRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim=50, hidden_size=64):
        super().__init__()

        # স্তর ১: Word Embedding — প্রতিটি word index → 50-dimensional vector
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # স্তর ২: RNN — sequence process করে একটি final hidden state বের করে
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

        # স্তর ৩: Fully Connected — hidden state থেকে vocab size-এর output
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, question):
        embedded = self.embedding(question)   # shape: (1, seq_len, 50)
        _, final = self.rnn(embedded)         # final shape: (1, 1, 64)

        return self.fc(final.squeeze(0))      # shape: (1, vocab_size)
```

```python
model = simpleRNN(vocab_size=len(vocab))
```

### 🔍 ব্যাখ্যা

Model-টিতে মোট **৩টি স্তর (layer)** আছে:

#### স্তর ১ — Embedding Layer (`nn.Embedding`)
```
Input:  [1, 2, 5, 3]        (word indices, shape: seq_len)
Output: [[v1], [v2], [v5], [v3]]  (embedding vectors, shape: seq_len × 50)
```
প্রতিটি শব্দের index-কে একটি 50-dimensional শেখার উপযোগী vector-এ রূপান্তর করে। এই vector-গুলো training-এর সময় model নিজেই শেখে।

#### স্তর ২ — RNN Layer (`nn.RNN`)
```
Input:  sequence of embedding vectors (seq_len × 50)
Output: _, final_hidden_state  (shape: 1 × 1 × 64)
```
- RNN প্রতিটি শব্দ একে একে process করে এবং একটি "মেমোরি" (hidden state) বহন করে।
- `_` হলো সব time step-এর output (আমরা এটা ব্যবহার করছি না)।
- `final` হলো শেষ hidden state — পুরো প্রশ্নের "সারমর্ম"।
- `batch_first=True` মানে input-এর প্রথম dimension হবে batch।
#### স্তর ৩ — Linear Layer (`nn.Linear`)
```
Input:  final hidden state (64-dimensional)
Output: logits (vocab_size-dimensional)
```
64-dimensional hidden state থেকে vocab-এর প্রতিটি শব্দের জন্য একটি score (logit) তৈরি করে। সর্বোচ্চ score-এর শব্দটিই হবে predicted উত্তর।

#### Forward Pass সংক্ষেপে:
```
question indices → Embedding → RNN → squeeze → Linear → logits (vocab_size)
```

---